In [1]:
import sys
import subprocess

import os
import time
import warnings
warnings.filterwarnings('ignore')

os.environ['DDE_BACKEND'] = 'pytorch'
os.makedirs('results_2d', exist_ok=True)
os.makedirs('models_2d', exist_ok=True)

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import datetime

import deepxde as dde
import torch

dde.config.set_default_float('float32')


plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.25,
    'figure.dpi':        140,
})

G        = 9.81
X_MIN    = -5.0
X_MAX    =  5.0
Y_MIN    = -5.0
Y_MAX    =  5.0
T_FINAL  = 2.0

DAM_RADIUS = 1.5
H_INSIDE   = 4.0
H_OUTSIDE  = 1.0

NUM_DOMAIN   = 8000
NUM_BOUNDARY = 1000
NUM_INITIAL  = 1500
NUM_TEST     = 10000

ADAM_ITER_PHASE1 = 8000
ADAM_ITER_PHASE2 = 6000
LR_PHASE1        = 1e-3
LR_PHASE2        = 3e-4

W_PDE_1 = 1.0
W_IC_1  = 15.0
W_BC_1  = 2.0
W_PDE_2 = 3.0
W_IC_2  = 5.0
W_BC_2  = 2.0
W_POS   = 200.0
NU_ART  = 0.002

nx_plot = 100
xp = np.linspace(X_MIN, X_MAX, nx_plot)
yp = np.linspace(Y_MIN, Y_MAX, nx_plot)
Xp, Yp = np.meshgrid(xp, yp)
r      = np.sqrt(Xp**2 + Yp**2)
H_init = np.where(r < DAM_RADIUS, H_INSIDE, H_OUTSIDE)

fig, ax = plt.subplots(figsize=(6, 6))
cf = ax.contourf(Xp, Yp, H_init, levels=50, cmap='Blues')
plt.colorbar(cf, ax=ax, label='h [m]')
circle = plt.Circle((0, 0), DAM_RADIUS, color='red',
                     fill=False, lw=2, ls='--', label=f'Dam (r={DAM_RADIUS}m)')
ax.add_patch(circle)
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('Initial condition h(x,y,0)', fontweight='bold')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('results_2d/initial_condition.png', dpi=150, bbox_inches='tight')
plt.close()


def swe2d_pde(x, y):
    h  = y[:, 0:1]
    qx = y[:, 1:2]
    qy = y[:, 2:3]

    h_safe = h + 1e-6
    u = qx / h_safe
    v = qy / h_safe

    h_t  = dde.grad.jacobian(y, x, i=0, j=2)
    qx_t = dde.grad.jacobian(y, x, i=1, j=2)
    qy_t = dde.grad.jacobian(y, x, i=2, j=2)

    qx_x = dde.grad.jacobian(y, x, i=1, j=0)
    qy_y = dde.grad.jacobian(y, x, i=2, j=1)

    h_x    = dde.grad.jacobian(y, x, i=0, j=0)
    uqx_x  = dde.grad.jacobian(u * qx, x, i=0, j=0)
    vqx_y  = dde.grad.jacobian(v * qx, x, i=0, j=1)
    gh_hx  = G * h * h_x

    h_y    = dde.grad.jacobian(y, x, i=0, j=1)
    uqy_x  = dde.grad.jacobian(u * qy, x, i=0, j=0)
    vqy_y  = dde.grad.jacobian(v * qy, x, i=0, j=1)
    gh_hy  = G * h * h_y

    h_xx  = dde.grad.hessian(y, x, component=0, i=0, j=0)
    h_yy  = dde.grad.hessian(y, x, component=0, i=1, j=1)
    qx_xx = dde.grad.hessian(y, x, component=1, i=0, j=0)
    qx_yy = dde.grad.hessian(y, x, component=1, i=1, j=1)
    qy_xx = dde.grad.hessian(y, x, component=2, i=0, j=0)
    qy_yy = dde.grad.hessian(y, x, component=2, i=1, j=1)

    continuity = h_t  + qx_x + qy_y - NU_ART * (h_xx  + h_yy)
    x_momentum = qx_t + uqx_x + vqx_y + gh_hx - NU_ART * (qx_xx + qx_yy)
    y_momentum = qy_t + uqy_x + vqy_y + gh_hy - NU_ART * (qy_xx + qy_yy)
    pos_penalty = W_POS * torch.relu(-h)

    return [continuity, x_momentum, y_momentum, pos_penalty]


def ic_h(x):
    r = np.sqrt(x[:, 0:1]**2 + x[:, 1:2]**2)
    return np.where(r < DAM_RADIUS, H_INSIDE, H_OUTSIDE).astype(np.float32)

def ic_qx(x):
    return np.zeros((x.shape[0], 1), dtype=np.float32)

def ic_qy(x):
    return np.zeros((x.shape[0], 1), dtype=np.float32)


def setup_problem():
    geom     = dde.geometry.Rectangle([X_MIN, Y_MIN], [X_MAX, Y_MAX])
    time_dom = dde.geometry.TimeDomain(0, T_FINAL)
    geomtime = dde.geometry.GeometryXTime(geom, time_dom)

    IC_h  = dde.icbc.IC(geomtime, ic_h,  lambda _, on_ic: on_ic, component=0)
    IC_qx = dde.icbc.IC(geomtime, ic_qx, lambda _, on_ic: on_ic, component=1)
    IC_qy = dde.icbc.IC(geomtime, ic_qy, lambda _, on_ic: on_ic, component=2)

    def on_boundary(x, on_b): return on_b

    BC_h = dde.icbc.NeumannBC(geomtime, lambda x: 0, on_boundary, component=0)

    data = dde.data.TimePDE(
        geomtime,
        swe2d_pde,
        [IC_h, IC_qx, IC_qy, BC_h],
        num_domain   = NUM_DOMAIN,
        num_boundary = NUM_BOUNDARY,
        num_initial  = NUM_INITIAL,
        num_test     = NUM_TEST,
    )
    return data

data = setup_problem()


def build_network():
    net = dde.nn.FNN(
        layer_sizes        = [3] + [64] * 6 + [3],
        activation         = 'tanh',
        kernel_initializer = 'Glorot normal',
    )

    L = X_MAX - X_MIN
    net.apply_feature_transform(
        lambda x: torch.stack([
            2 * (x[:, 0] - X_MIN) / L - 1,
            2 * (x[:, 1] - Y_MIN) / L - 1,
            2 *  x[:, 2] / T_FINAL    - 1,
        ], dim=1)
    )

    net.apply_output_transform(
        lambda x, y: torch.stack([
            torch.nn.functional.softplus(y[:, 0]),
            y[:, 1],
            y[:, 2],
        ], dim=1)
    )

    return net

net   = build_network()
model = dde.Model(data, net)

total_params = sum(p.numel() for p in net.parameters())
print(f'Parameters: {total_params:,}')


def reference_solver_2d(nx=80, ny=80, nt_save=50, nu=0.01):
    dx = (X_MAX - X_MIN) / nx
    dy = (Y_MAX - Y_MIN) / ny

    x1d = np.linspace(X_MIN + dx/2, X_MAX - dx/2, nx)
    y1d = np.linspace(Y_MIN + dy/2, Y_MAX - dy/2, ny)
    Xg, Yg = np.meshgrid(x1d, y1d)

    r  = np.sqrt(Xg**2 + Yg**2)
    h  = np.where(r < DAM_RADIUS, H_INSIDE, H_OUTSIDE).astype(float)
    qx = np.zeros_like(h)
    qy = np.zeros_like(h)

    t       = 0.0
    t_saves = np.linspace(0, T_FINAL, nt_save)
    h_out, qx_out, qy_out, t_out = [], [], [], []
    snap_idx = 0

    while t < T_FINAL + 1e-10:
        h_s  = np.maximum(h, 1e-6)
        u    = qx / h_s
        v    = qy / h_s
        c    = np.sqrt(G * h_s)
        spd  = np.sqrt(u**2 + v**2)
        dt   = 0.4 * min(dx, dy) / np.max(spd + c + 1e-10)
        dt   = min(dt, T_FINAL - t + 1e-12)
        if dt <= 0:
            break

        def ghost(f):
            return np.pad(f, 1, mode='edge')

        hg  = ghost(h);  qxg = ghost(qx); qyg = ghost(qy)

        def Fh_x(q_):         return q_
        def Fqx_x(h_, q_, qs_): return qs_ * q_ / np.maximum(h_, 1e-6) + 0.5*G*h_**2
        def Fqy_x(h_, q_, qs_): return qs_ * q_ / np.maximum(h_, 1e-6)

        hL  = hg[1:-1, :-2];  hR  = hg[1:-1, 2:]
        qxL = qxg[1:-1, :-2]; qxR = qxg[1:-1, 2:]
        qyL = qyg[1:-1, :-2]; qyR = qyg[1:-1, 2:]

        h_new_x  = 0.5*(hL+hR)   - dt/(2*dx)*(Fh_x(qxR)               - Fh_x(qxL))
        qx_new_x = 0.5*(qxL+qxR) - dt/(2*dx)*(Fqx_x(hR,qxR,qxR)      - Fqx_x(hL,qxL,qxL))
        qy_new_x = 0.5*(qyL+qyR) - dt/(2*dx)*(Fqy_x(hR,qyR,qxR)      - Fqy_x(hL,qyL,qxL))

        hg2  = ghost(h_new_x)
        qxg2 = ghost(qx_new_x)
        qyg2 = ghost(qy_new_x)

        hB  = hg2[:-2, 1:-1];  hT  = hg2[2:, 1:-1]
        qxB = qxg2[:-2, 1:-1]; qxT = qxg2[2:, 1:-1]
        qyB = qyg2[:-2, 1:-1]; qyT = qyg2[2:, 1:-1]

        h_new  = 0.5*(hB+hT)   - dt/(2*dy)*(qyT                       - qyB)
        qx_new = 0.5*(qxB+qxT) - dt/(2*dy)*(Fqy_x(hT,qxT,qyT)        - Fqy_x(hB,qxB,qyB))
        qy_new = 0.5*(qyB+qyT) - dt/(2*dy)*(Fqx_x(hT,qyT,qyT)        - Fqx_x(hB,qyB,qyB))

        hg3  = ghost(h);  qxg3 = ghost(qx); qyg3 = ghost(qy)

        lap_h  = (hg3[1:-1,:-2] + hg3[1:-1,2:] + hg3[:-2,1:-1] + hg3[2:,1:-1] - 4*h)   / (dx*dy)
        lap_qx = (qxg3[1:-1,:-2] + qxg3[1:-1,2:] + qxg3[:-2,1:-1] + qxg3[2:,1:-1] - 4*qx) / (dx*dy)
        lap_qy = (qyg3[1:-1,:-2] + qyg3[1:-1,2:] + qyg3[:-2,1:-1] + qyg3[2:,1:-1] - 4*qy) / (dx*dy)

        h_new  += nu * dt * lap_h
        qx_new += nu * dt * lap_qx
        qy_new += nu * dt * lap_qy
        h_new   = np.maximum(h_new, 0.0)

        h, qx, qy = h_new, qx_new, qy_new
        t += dt

        if snap_idx < nt_save and t >= t_saves[snap_idx] - 1e-9:
            h_out.append(h.copy())
            qx_out.append(qx.copy())
            qy_out.append(qy.copy())
            t_out.append(t)
            snap_idx += 1

    return (x1d, y1d, Xg, Yg,
            np.array(t_out),
            np.array(h_out),
            np.array(qx_out),
            np.array(qy_out))


print('Running reference solver...')
t0 = time.time()
x1d, y1d, Xg, Yg, t_saves, h_ref, qx_ref, qy_ref = reference_solver_2d(nx=80, ny=80, nt_save=50)
ref_time = time.time() - t0
print(f'Reference solver done in {ref_time:.2f}s')

idxs = [0, 12, 24, 37, 49]
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle('Reference solution — radial wave', fontsize=13, fontweight='bold', y=1.02)

vmin = h_ref.min(); vmax = h_ref.max()
for j, (ax, idx) in enumerate(zip(axes, idxs)):
    cf = ax.contourf(Xg, Yg, h_ref[idx], levels=40, cmap='viridis', vmin=vmin, vmax=vmax)
    ax.set_title(f't = {t_saves[idx]:.2f}s', fontweight='bold')
    ax.set_xlabel('x [m]')
    ax.set_aspect('equal')
    if j == 0:
        ax.set_ylabel('y [m]')
        plt.colorbar(cf, ax=ax, label='h [m]')

plt.tight_layout()
plt.savefig('results_2d/reference_snapshots.png', dpi=150, bbox_inches='tight')
plt.close()

ckpt = dde.callbacks.ModelCheckpoint(
    'models_2d/ckpt',
    save_better_only=False,
    period=3000,
)

print(f'Training started: {datetime.now().strftime("%H:%M:%S")}')
print('Phase 1: Adam IC-heavy...')

model.compile(
    'adam', lr=LR_PHASE1,
    loss_weights=[
        W_PDE_1, W_PDE_1, W_PDE_1, W_POS,
        W_IC_1,  W_IC_1,  W_IC_1,
        W_BC_1,
    ],
)
losshistory1, _ = model.train(
    iterations=ADAM_ITER_PHASE1,
    display_every=2000,
    callbacks=[ckpt],
)
print(f'Phase 1 done: {datetime.now().strftime("%H:%M:%S")}')

print('Phase 2: Adam PDE-heavy...')
model.compile(
    'adam', lr=LR_PHASE2,
    loss_weights=[
        W_PDE_2, W_PDE_2, W_PDE_2, W_POS,
        W_IC_2,  W_IC_2,  W_IC_2,
        W_BC_2,
    ],
)
losshistory2, _ = model.train(
    iterations=ADAM_ITER_PHASE2,
    display_every=2000,
    callbacks=[ckpt],
)
print(f'Phase 2 done: {datetime.now().strftime("%H:%M:%S")}')

print('L-BFGS polish...')
model.compile('L-BFGS')
losshistory3, _ = model.train()
model.save('models_2d/swe2d_circular')
print(f'Training complete: {datetime.now().strftime("%H:%M:%S")}')

loss_arr = (
    np.array(losshistory1.loss_train).sum(axis=1).tolist() +
    np.array(losshistory2.loss_train).sum(axis=1).tolist() +
    np.array(losshistory3.loss_train).sum(axis=1).tolist()
)

plt.figure(figsize=(10, 4))
steps = np.arange(len(loss_arr))
plt.semilogy(steps, loss_arr, color='#2C3E50', lw=1, alpha=0.4)
win    = max(1, len(loss_arr)//60)
smooth = np.convolve(loss_arr, np.ones(win)/win, mode='valid')
plt.semilogy(np.arange(win//2, win//2+len(smooth)), smooth, color='#4A6CF7', lw=2.2, label='Smoothed')
plt.axvline(ADAM_ITER_PHASE1, color='#E94B6B', ls=':', lw=1.5, label='Phase 2 start')
plt.xlabel('Iteration'); plt.ylabel('Total loss')
plt.title('Training convergence', fontweight='bold')
plt.legend(); plt.tight_layout()
plt.savefig('results_2d/loss_curve.png', dpi=150, bbox_inches='tight')
plt.close()

print('Running inference...')
nx_ref = len(x1d)
ny_ref = len(y1d)
nt_ref = len(t_saves)

X3d, Y3d, T3d = np.meshgrid(x1d, y1d, t_saves, indexing='ij')
xyt_flat = np.column_stack([X3d.ravel(), Y3d.ravel(), T3d.ravel()]).astype(np.float32)

t0        = time.time()
pred      = model.predict(xyt_flat)
pinn_time = time.time() - t0

h_pinn  = pred[:, 0].reshape(nx_ref, ny_ref, nt_ref).transpose(2, 1, 0)
qx_pinn = pred[:, 1].reshape(nx_ref, ny_ref, nt_ref).transpose(2, 1, 0)
qy_pinn = pred[:, 2].reshape(nx_ref, ny_ref, nt_ref).transpose(2, 1, 0)

print(f'Inference done in {pinn_time*1000:.1f} ms')

err_h = np.abs(h_pinn - h_ref)
l2    = np.sqrt(np.mean(err_h**2))
linf  = err_h.max()
rel   = l2 / (np.sqrt(np.mean(h_ref**2)) + 1e-10)

dx_ref    = x1d[1] - x1d[0]
dy_ref    = y1d[1] - y1d[0]
mass_ref  = h_ref.sum(axis=(1,2))  * dx_ref * dy_ref
mass_pinn = h_pinn.sum(axis=(1,2)) * dx_ref * dy_ref
drift     = abs((mass_pinn[-1] - mass_pinn[0]) / mass_pinn[0]) * 100
speedup   = ref_time / max(pinn_time, 1e-9)

print('='*50)
print(f'  h  L2  error       {l2:.5f} m')
print(f'  h  L-inf error     {linf:.5f} m')
print(f'  h  relative L2     {rel:.5f}')
print(f'  Mass drift (PINN)  {drift:.4f} %')
print(f'  Speedup            x{speedup:.0f}')
print('='*50)

with open('results_2d/metrics.txt', 'w') as f:
    f.write(f'h L2 error:       {l2:.6e}\n')
    f.write(f'h Linf error:     {linf:.6e}\n')
    f.write(f'h rel L2:         {rel:.6e}\n')
    f.write(f'Mass drift:       {drift:.4f} %\n')
    f.write(f'Ref solver time:  {ref_time:.3f} s\n')
    f.write(f'PINN infer time:  {pinn_time*1000:.3f} ms\n')
    f.write(f'Speedup:          x{speedup:.0f}\n')

idxs = np.linspace(0, nt_ref-1, 5, dtype=int)
fig, axes = plt.subplots(3, 5, figsize=(22, 12))
fig.suptitle('2D Circular Dam Break — PINN vs Reference', fontsize=14, fontweight='bold')

vmin = min(h_pinn.min(), h_ref.min())
vmax = max(h_pinn.max(), h_ref.max())
emax = err_h.max()

for j, idx in enumerate(idxs):
    cf0 = axes[0, j].contourf(Xg, Yg, h_ref[idx], levels=40, cmap='viridis', vmin=vmin, vmax=vmax)
    axes[0, j].set_title(f't = {t_saves[idx]:.2f}s', fontweight='bold')
    axes[0, j].set_aspect('equal')

    cf1 = axes[1, j].contourf(Xg, Yg, h_pinn[idx], levels=40, cmap='viridis', vmin=vmin, vmax=vmax)
    axes[1, j].set_aspect('equal')

    cf2 = axes[2, j].contourf(Xg, Yg, err_h[idx], levels=40, cmap='Reds', vmin=0, vmax=emax)
    axes[2, j].set_aspect('equal')
    axes[2, j].set_xlabel('x [m]')

axes[0, 0].set_ylabel('Reference\ny [m]', fontweight='bold', color='#F76A4A')
axes[1, 0].set_ylabel('PINN\ny [m]', fontweight='bold', color='#4A6CF7')
axes[2, 0].set_ylabel('|Error|\ny [m]', fontweight='bold', color='#E94B6B')

plt.colorbar(cf0, ax=axes[0, -1], label='h [m]')
plt.colorbar(cf1, ax=axes[1, -1], label='h [m]')
plt.colorbar(cf2, ax=axes[2, -1], label='|error| [m]')

plt.tight_layout()
plt.savefig('results_2d/comparison_grid.png', dpi=150, bbox_inches='tight')
plt.close()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Training & Physical Consistency', fontweight='bold')

steps = np.arange(len(loss_arr))
ax1.semilogy(steps, loss_arr, color='#2C3E50', lw=1, alpha=0.4)
win    = max(1, len(loss_arr)//60)
smooth = np.convolve(loss_arr, np.ones(win)/win, mode='valid')
ax1.semilogy(np.arange(win//2, win//2+len(smooth)), smooth, color='#4A6CF7', lw=2.2, label='Smoothed')
ax1.axvline(ADAM_ITER_PHASE1, color='#E94B6B', ls=':', lw=1.2, label='Phase 2 start')
ax1.set_xlabel('Iteration'); ax1.set_ylabel('Total Loss')
ax1.set_title('Loss Convergence'); ax1.legend()

dr = 100*(mass_ref  - mass_ref[0])  / mass_ref[0]
dp = 100*(mass_pinn - mass_pinn[0]) / mass_pinn[0]
ax2.plot(t_saves, dr, color='#F76A4A', lw=2, label='Reference')
ax2.plot(t_saves, dp, color='#4A6CF7', lw=2, ls='--', label='PINN')
ax2.axhline(0, color='gray', lw=0.8, ls=':')
ax2.set_xlabel('t [s]'); ax2.set_ylabel('Mass drift [%]')
ax2.set_title('2D Mass Conservation'); ax2.legend()

plt.tight_layout()
plt.savefig('results_2d/loss_and_conservation.png', dpi=150, bbox_inches='tight')
plt.close()

t_target = 1.0
idx      = np.argmin(np.abs(t_saves - t_target))

h_s  = np.maximum(h_pinn[idx], 1e-6)
u_pl = qx_pinn[idx] / h_s
v_pl = qy_pinn[idx] / h_s

step = 8
Xs = Xg[::step, ::step]
Ys = Yg[::step, ::step]
Us = u_pl[::step, ::step]
Vs = v_pl[::step, ::step]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Velocity field at t = {t_saves[idx]:.2f}s', fontsize=13, fontweight='bold')

cf = axes[0].contourf(Xg, Yg, h_pinn[idx], levels=40, cmap='Blues', alpha=0.7)
axes[0].quiver(Xs, Ys, Us, Vs, color='#E94B6B', scale=30, width=0.003)
plt.colorbar(cf, ax=axes[0], label='h [m]')
axes[0].set_title('PINN', fontweight='bold')
axes[0].set_xlabel('x [m]'); axes[0].set_ylabel('y [m]')
axes[0].set_aspect('equal')

h_rs  = np.maximum(h_ref[idx], 1e-6)
u_ref = qx_ref[idx] / h_rs
v_ref = qy_ref[idx] / h_rs
Ur = u_ref[::step, ::step]
Vr = v_ref[::step, ::step]

cf2 = axes[1].contourf(Xg, Yg, h_ref[idx], levels=40, cmap='Blues', alpha=0.7)
axes[1].quiver(Xs, Ys, Ur, Vr, color='#E94B6B', scale=30, width=0.003)
plt.colorbar(cf2, ax=axes[1], label='h [m]')
axes[1].set_title('Reference', fontweight='bold')
axes[1].set_xlabel('x [m]'); axes[1].set_ylabel('y [m]')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.savefig('results_2d/velocity_field.png', dpi=150, bbox_inches='tight')
plt.close()

x_val = 2.0
y_val = 0.0
t_val = 1.0

xyt  = np.array([[x_val, y_val, t_val]], dtype=np.float32)
t0   = time.perf_counter()
pred = model.predict(xyt)
us   = (time.perf_counter() - t0) * 1e6

h_q  = float(pred[0, 0])
qx_q = float(pred[0, 1])
qy_q = float(pred[0, 2])
u_q  = qx_q / max(h_q, 1e-6)
v_q  = qy_q / max(h_q, 1e-6)
spd  = np.sqrt(u_q**2 + v_q**2)

print(f'Query  : x={x_val}m,  y={y_val}m,  t={t_val}s')
print(f'h  = {h_q:.4f} m')
print(f'qx = {qx_q:.4f} m2/s')
print(f'qy = {qy_q:.4f} m2/s')
print(f'u  = {u_q:.4f} m/s')
print(f'v  = {v_q:.4f} m/s')
print(f'speed = {spd:.4f} m/s')
print(f'Inference time = {us:.1f} us')

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Set the default float type to float32
Parameters: 21,251
Running reference solver...
Reference solver done in 0.31s
Training started: 03:30:47
Phase 1: Adam IC-heavy...
Compiling model...
'compile' took 2.956734 s

Training model...

Step      Train loss                                                                          Test loss                                                                           Test metric
0         [5.22e-04, 2.37e-02, 2.20e-02, 0.00e+00, 1.22e+01, 8.56e-01, 8.47e-01, 1.50e-04]    [5.16e-04, 2.72e-02, 2.39e-02, 0.00e+00, 1.22e+01, 8.56e-01, 8.47e-01, 1.50e-04]    []  
2000      [1.72e-01, 3.98e-02, 3.75e-02, 0.00e+00, 4.09e-01, 5.86e-03, 5.71e-03, 3.52e-02]    [1.99e-01, 1.07e-01, 1.32e-01, 0.00e+00, 4.09e-01, 5.86e-03, 5.71e-03, 3.52e-02]    []  
4000      [1.32e-01, 3.03e-02, 3.04e-02, 0.00e+00, 2.48e-01, 3.14e-03, 3.46e-03, 3.32e-02]    [1.99e-01, 1.81e-01, 2.86e-01, 0.00e+00, 2.48e-01, 3.14e-03, 3.46e-03, 3.32e-02]    []  
6000      [1.14e-01, 2.51e-